## Langkah 1: Persiapan Awal (Konfigurasi & Setup Database)

In [56]:
import sqlite3
import os

BASE_DIR = os.getcwd()
NAMA_DB = "pengeluaran_harian.db"
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)

def setup_database():
    conn = sqlite3.connect(DB_PATH)

    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS transaksi (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        deskripsi TEXT NOT NULL,
        jumlah REAL NOT NULL,
        kategori TEXT,
        tanggal DATE NOT NULL
    )
    """)

    conn.commit()
    conn.close()

    print("Database siap")

setup_database()

Database siap


## Langkah 2: Modul Akses Database (database.py)

In [57]:
import sqlite3
import pandas as pd

def get_db_connection():
    return sqlite3.connect(DB_PATH)

def execute_query(query, params=None):

    conn = get_db_connection()

    cursor = conn.cursor()

    if params:
        cursor.execute(query, params)
    else:
        cursor.execute(query)

    conn.commit()

    last_id = cursor.lastrowid

    conn.close()

    return last_id

def fetch_query(query, params=None):

    conn = get_db_connection()

    cursor = conn.cursor()

    if params:
        cursor.execute(query, params)
    else:
        cursor.execute(query)

    result = cursor.fetchall()

    conn.close()

    return result

def get_dataframe(query):

    conn = get_db_connection()

    df = pd.read_sql_query(query, conn)

    conn.close()

    return df

## Langkah 3: Modul Model Data (model.py)

In [58]:
import datetime

class Transaksi:

    def __init__(
        self,
        deskripsi,
        jumlah,
        kategori,
        tanggal,
        id_transaksi=None
    ):

        self.id = id_transaksi
        self.deskripsi = deskripsi
        self.jumlah = float(jumlah)
        self.kategori = kategori

        if isinstance(tanggal, str):
            self.tanggal = datetime.datetime.strptime(
                tanggal,
                "%Y-%m-%d"
            ).date()
        else:
            self.tanggal = tanggal

    def to_dict(self):

        return {
            "deskripsi": self.deskripsi,
            "jumlah": self.jumlah,
            "kategori": self.kategori,
            "tanggal": self.tanggal.strftime("%Y-%m-%d")
        }

    def __repr__(self):

        return (
            f"{self.deskripsi} | "
            f"{self.kategori} | "
            f"Rp {self.jumlah:,.0f}"
        )

## Langkah 4: Modul Manajer Anggaran (manajer_anggaran.py)

In [59]:
class AnggaranHarian:

    def tambah_transaksi(self, transaksi):

        sql = """
        INSERT INTO transaksi
        (deskripsi, jumlah, kategori, tanggal)
        VALUES (?, ?, ?, ?)
        """

        params = (
            transaksi.deskripsi,
            transaksi.jumlah,
            transaksi.kategori,
            transaksi.tanggal.strftime("%Y-%m-%d")
        )

        execute_query(sql, params)

        return True

    def tampilkan_transaksi(self):

        sql = """
        SELECT *
        FROM transaksi
        ORDER BY id DESC
        """

        return get_dataframe(sql)

    def total_pengeluaran(self):

        sql = """
        SELECT SUM(jumlah)
        FROM transaksi
        """

        hasil = fetch_query(sql)

        return hasil[0][0] if hasil[0][0] else 0

    def hapus_transaksi(self, id_transaksi):

        sql = """
        DELETE FROM transaksi
        WHERE id = ?
        """

        execute_query(sql, (id_transaksi,))

        return True

## Langkah 5: Aplikasi Utama Streamlit (main_app.py)

In [70]:
import datetime

anggaran = AnggaranHarian()

anggaran.tambah_transaksi(
    Transaksi(
        "Naik Ojol",
        15000,
        "Transportasi",
        datetime.date.today()
    )
)

anggaran.tambah_transaksi(
    Transaksi(
        "Beli Buku",
        50000,
        "Pendidikan",
        datetime.date.today()
    )
)

True

## Langkah 6: Menjalankan dan Menguji Aplikasi Modular

In [71]:
print("Total Pengeluaran:")
print(
    anggaran.total_pengeluaran()
)

print("\nData Transaksi:")

display(
    anggaran.tampilkan_transaksi()
)

Total Pengeluaran:
65000.0

Data Transaksi:


,id,deskripsi,jumlah,kategori,tanggal
0,20,Beli Buku,50000.0,Pendidikan,2026-06-21
1,19,Naik Ojol,15000.0,Transportasi,2026-06-21


## Penugasan

In [73]:
anggaran = AnggaranHarian()

print("Data Sebelum Hapus:")
display(
    anggaran.tampilkan_transaksi()
)

# Hapus salah satu transaksi berdasarkan ID
anggaran.hapus_transaksi(19)

print("Data Setelah Hapus:")
display(
    anggaran.tampilkan_transaksi()
)

print("Total Pengeluaran Setelah Hapus:")
print(
    anggaran.total_pengeluaran()
)

Data Sebelum Hapus:


,id,deskripsi,jumlah,kategori,tanggal
0,20,Beli Buku,50000.0,Pendidikan,2026-06-21
1,19,Naik Ojol,15000.0,Transportasi,2026-06-21


Data Setelah Hapus:


,id,deskripsi,jumlah,kategori,tanggal
0,20,Beli Buku,50000.0,Pendidikan,2026-06-21


Total Pengeluaran Setelah Hapus:
50000.0
